In [9]:
!pip install xgboost


   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
    --------------------------------------- 1.0/72.0 MB 3.4 MB/s eta 0:00:21
    --------------------------------------- 1.6/72.0 MB 3.1 MB/s eta 0:00:23
   - -------------------------------------- 2.4/72.0 MB 3.2 MB/s eta 0:00:22
   - -------------------------------------- 2.9/72.0 MB 3.1 MB/s eta 0:00:23
   - -------------------------------------- 3.4/72.0 MB 3.2 MB/s eta 0:00:22
   -- ------------------------------------- 4.2/72.0 MB 3.2 MB/s eta 0:00:22
   -- ------------------------------------- 5.0/72.0 MB 3.2 MB/s eta 0:00:21
   --- ------------------------------------ 5.8/72.0 MB 3.3 MB/s eta 0:00:21
   --- ------------------------------------ 6.6/72.0 MB 3.3 MB/s eta 0:00:20
   ---- ----------------------------------- 7.3/72.0 MB 3.4 MB/s eta 0:00:20
   ---- ----------------------------------- 7.9/72.0 MB 3.4 MB/s eta 0:00:19
   ---- -----


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
!pip install --upgrade xgboost



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [15]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [16]:
df_train

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,97075632,1,0.000000,1,51.295195,0.000000,0.000000,0.000000,3.956904,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,17,6,6,66.340624,0
217,97362560,2,1.127102,0,1.176991,0.000000,0.000000,0.000000,0.777943,0.000000,...,0.000000,-0.277779,0.277779,-0.000000,-0.000000,18,1,7,5.694898,1
218,97805715,2,3.710653,0,71.946930,0.000000,0.000000,0.000000,4.289732,0.000000,...,0.000000,0.694609,0.694609,0.000000,0.000000,18,1,9,44.011253,0
219,99071478,1,0.000000,1,20.223659,0.000000,0.000000,0.000000,3.055117,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,15,0,8,22.975783,1


In [17]:
df_test

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month
0,10662602,1,0.000000,1,2.452217,0.000000,0.00000,0.000000,1.239017,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,3,7
1,13353600,1,0.000000,1,131.669588,0.000000,0.00000,0.000000,4.887862,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22,0,8
2,13942327,1,0.000000,1,6.723104,0.000000,0.00000,0.000000,2.044216,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,6,7
3,16112781,1,0.000000,1,285.416736,0.000000,0.00000,0.000000,5.657448,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,7
4,17132808,7,3.459331,0,61.098604,12.516633,0.20486,3.618224,4.128724,2.603921,...,13.54413,-22.687575,0.044572,0.158550,0.158550,-24.414806,3.920562,23,5,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,94627327,2,4.614923,0,24.438326,0.000000,0.00000,0.000000,3.236257,0.000000,...,0.00000,0.000000,0.000000,0.763809,0.763809,0.000000,0.000000,22,0,6
91,96570675,1,0.000000,1,155.843418,0.000000,0.00000,0.000000,5.055248,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21,2,7
92,97225766,1,0.000000,1,87.761553,0.000000,0.00000,0.000000,4.485954,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4,1,7
93,98446281,2,4.335964,0,11.391108,0.000000,0.00000,0.000000,2.516979,0.000000,...,0.00000,0.000000,0.000000,-0.305683,0.305683,0.000000,-0.000000,17,0,9


In [18]:
test_ids = df_test['event_id'].copy()

In [19]:
drop_cols = ['event', 'time_to_hit_hours', 'event_id']

X_train = df_train.drop(columns=drop_cols)
y_train = df_train['event']

X_test = df_test.drop(columns=drop_cols, errors='ignore')

In [20]:
from xgboost import XGBClassifier

In [30]:
model = XGBClassifier(
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5, 
    gamma=0.1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)


In [36]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# --- 1. Загрузка и подготовка (твое начало) ---
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
test_ids = df_test['event_id'].copy()
drop_cols = ['event', 'time_to_hit_hours', 'event_id']

X = df_train.drop(columns=drop_cols)
y = df_train['event']
X_test = df_test.drop(columns=drop_cols, errors='ignore')

# --- 2. Обработка категориальных признаков ---
# XGBoost любит формат 'category' для нечисловых колонок
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_features:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# --- 3. Кросс-валидация и обучение (Максимальная мощность) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
base_prob = np.zeros(len(X_test))

# Параметры под 24GB RAM и высокую точность
xgb_params = {
    'n_estimators': 3000,
    'learning_rate': 0.02, # Медленный шаг для точности
    'max_depth': 7,        # Оптимально, чтобы не переобучиться
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist', # Быстрый метод для больших данных
    'enable_categorical': True, 
    'random_state': 42,
    'n_jobs': -1,          # Используем все ядра процессора
}

print("Starting Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=100)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    base_prob += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"Fold {fold+1} AUC: {fold_auc:.5f}")

print(f"\nOverall OOF AUC: {roc_auc_score(y, oof_preds):.5f}")

# --- 4. Финализация (твой конец) ---
prob_12h = np.clip(base_prob * 0.6, 0, 1)
prob_24h = np.clip(base_prob * 0.8, 0, 1)
prob_48h = np.clip(base_prob * 1.0, 0, 1)
prob_72h = np.clip(base_prob * 1.2, 0, 1)

# Гарантируем монотонность (вероятность не может убывать со временем)
prob_24h = np.maximum(prob_24h, prob_12h)
prob_48h = np.maximum(prob_48h, prob_24h)
prob_72h = np.maximum(prob_72h, prob_48h)

df_sub = pd.DataFrame({
    'event_id': test_ids,
    'prob_12h': prob_12h,
    'prob_24h': prob_24h,
    'prob_48h': prob_48h,
    'prob_72h': prob_72h
})

df_sub.to_csv('XGBoost_sub.csv', index=False)

Starting Cross-Validation...
[0]	validation_0-logloss:0.60180
[100]	validation_0-logloss:0.10289
[200]	validation_0-logloss:0.02869
[300]	validation_0-logloss:0.01405
[400]	validation_0-logloss:0.01406
[405]	validation_0-logloss:0.01406
Fold 1 AUC: 1.00000
[0]	validation_0-logloss:0.59150
[100]	validation_0-logloss:0.16263
[200]	validation_0-logloss:0.11786
[300]	validation_0-logloss:0.12323
Fold 2 AUC: 0.99504
[0]	validation_0-logloss:0.60714
[100]	validation_0-logloss:0.10930
[200]	validation_0-logloss:0.03233
[300]	validation_0-logloss:0.01606
[400]	validation_0-logloss:0.01603
[500]	validation_0-logloss:0.01601
[600]	validation_0-logloss:0.01554
[650]	validation_0-logloss:0.01554
Fold 3 AUC: 1.00000
[0]	validation_0-logloss:0.60719
[100]	validation_0-logloss:0.10413
[200]	validation_0-logloss:0.02932
[300]	validation_0-logloss:0.01451
[400]	validation_0-logloss:0.01423
[500]	validation_0-logloss:0.01410
[526]	validation_0-logloss:0.01410
Fold 4 AUC: 1.00000
[0]	validation_0-logloss